# Exploring Categories with `cat.explore()`

Before classifying text, you need to know what categories exist in your data. `cat.explore()` discovers categories by sending chunks of your data to an LLM and collecting every category it identifies — **with duplicates intact**.

This raw output is ideal for **frequency analysis** (which categories are robust vs noise) and **saturation curves** (when has the model found all major themes).

**What this notebook covers:**
- How `explore()` works (chunking, iterations, raw output)
- Frequency analysis from raw categories
- Building saturation curves
- Controlling granularity with `specificity` and `focus`
- Research context, reproducibility, rate limiting
- Switching providers and using local models
- Saving results to CSV

**What you need:**
- Python 3.9+
- `pip install catllm`
- An API key from at least one supported provider

**See also:** `Extracting Categories with extract().ipynb` for the refined, deduplicated version.

---

## How `explore()` Works

1. **Divide** your data into `divisions` chunks (default 12)
2. **Send** each chunk to the LLM, asking it to identify up to `categories_per_chunk` categories
3. **Repeat** for `iterations` passes (default 8), reshuffling data each time
4. **Return** every category string from every chunk — duplicates included

The output length is approximately `iterations × divisions × categories_per_chunk`. Categories that appear many times across different chunks and iterations are robust themes; those appearing once or twice may be noise.

In [ ]:
import catllm as cat
import pandas as pd
from collections import Counter

## Setup

In [ ]:
# Replace with your actual API key
api_key = "YOUR_API_KEY"

## Sample Data

In [ ]:
responses = [
    "because i dont like living here",
    "for a bigger house",
    "to be with my wife",
    "got a new job in another state",
    "rent was too expensive so we had to leave",
    "my landlord sold the building so I had to find a new place",
    "wanted to be closer to my kids' school",
    "the neighborhood was getting unsafe",
    "retired and wanted to move somewhere warmer",
    "my partner got transferred to a different city",
    "needed more space after the baby was born",
    "commute was too long from where we were living",
    "moved in with my boyfriend",
    "lost my job and couldn't afford the mortgage",
    "just wanted a change of scenery honestly",
    "parents are getting older and need help",
    "flooding damaged our house and we had to relocate",
    "got accepted to graduate school out of state",
    "divorce, had to sell the house",
    "the apartment complex was being torn down for redevelopment",
]

## Basic Exploration

The simplest usage returns the raw list of every category discovered. We use fewer iterations and divisions here to keep API costs low.

In [ ]:
raw_categories = cat.explore(
    input_data=responses,
    description="Why did you move to your current residence?",
    api_key=api_key,
    user_model="gpt-4o-mini",
    iterations=3,
    divisions=4,
    categories_per_chunk=8,
)

print(f"Total raw categories returned: {len(raw_categories)}")
print(f"\nFirst 10:")
for i, c in enumerate(raw_categories[:10], 1):
    print(f"  {i}. {c}")

## Frequency Analysis

Since `explore()` preserves duplicates, you can count how often each category appears. Categories discovered repeatedly across different chunks and iterations are **robust** — the model consistently identifies them. Categories appearing only once or twice may be noise.

In [ ]:
counts = Counter(raw_categories)

print(f"Unique categories: {len(counts)}")
print(f"\nTop 15 by frequency:")
for category, freq in counts.most_common(15):
    print(f"  {freq:3d}x  {category}")

## Saturation Curves

A **saturation curve** tracks how the number of unique categories grows as you process more chunks. When the curve flattens, you've likely found all major themes — additional iterations won't discover new categories.

Run with more iterations to build a meaningful curve.

In [ ]:
raw_many = cat.explore(
    input_data=responses,
    description="Why did you move to your current residence?",
    api_key=api_key,
    user_model="gpt-4o-mini",
    iterations=8,
    divisions=4,
    categories_per_chunk=8,
)

# Build cumulative unique count
seen = set()
cumulative = []
for cat_name in raw_many:
    seen.add(cat_name)
    cumulative.append(len(seen))

print(f"Total raw: {len(raw_many)}, Unique: {len(seen)}")
print(f"\nCumulative unique categories at every 20th entry:")
for i in range(0, len(cumulative), 20):
    print(f"  After {i+1:3d} entries: {cumulative[i]} unique categories")
print(f"  After {len(cumulative):3d} entries: {cumulative[-1]} unique categories")

## Controlling Granularity: `specificity`

- **`"broad"`** (default): High-level themes (e.g., "Financial reasons")
- **`"specific"`**: Granular categories (e.g., "Rent increase", "Mortgage affordability")

In [ ]:
raw_broad = cat.explore(
    input_data=responses,
    description="Why did you move to your current residence?",
    api_key=api_key,
    user_model="gpt-4o-mini",
    specificity="broad",
    iterations=3,
    divisions=4,
)

raw_specific = cat.explore(
    input_data=responses,
    description="Why did you move to your current residence?",
    api_key=api_key,
    user_model="gpt-4o-mini",
    specificity="specific",
    iterations=3,
    divisions=4,
)

print(f"Broad: {len(set(raw_broad))} unique categories")
for cat_name, freq in Counter(raw_broad).most_common(8):
    print(f"  {freq:2d}x  {cat_name}")

print(f"\nSpecific: {len(set(raw_specific))} unique categories")
for cat_name, freq in Counter(raw_specific).most_common(8):
    print(f"  {freq:2d}x  {cat_name}")

## Steering Discovery with `focus`

The `focus` parameter tells the model what aspects to emphasize. Useful when your data covers many topics but you only care about a subset.

In [ ]:
raw_focused = cat.explore(
    input_data=responses,
    description="Why did you move to your current residence?",
    focus="financial and economic motivations for moving",
    api_key=api_key,
    user_model="gpt-4o-mini",
    iterations=3,
    divisions=4,
)

print(f"Focused categories ({len(set(raw_focused))} unique):")
for cat_name, freq in Counter(raw_focused).most_common(10):
    print(f"  {freq:2d}x  {cat_name}")

## Research Context

`research_question` provides broader academic context. While `description` tells the model *what* the data is, `research_question` tells it *why* you're analyzing it.

In [ ]:
raw_research = cat.explore(
    input_data=responses,
    description="Why did you move to your current residence?",
    research_question="What push and pull factors drive residential mobility among US adults?",
    api_key=api_key,
    user_model="gpt-4o-mini",
    iterations=3,
    divisions=4,
)

print(f"With research context ({len(set(raw_research))} unique):")
for cat_name, freq in Counter(raw_research).most_common(10):
    print(f"  {freq:2d}x  {cat_name}")

## Saving to CSV

Pass a `filename` to save the raw category list as a CSV with one category per row.

In [ ]:
raw_saved = cat.explore(
    input_data=responses,
    description="Why did you move to your current residence?",
    api_key=api_key,
    user_model="gpt-4o-mini",
    iterations=3,
    divisions=4,
    filename="raw_categories.csv",
)

saved_df = pd.read_csv("raw_categories.csv")
print(f"Saved {len(saved_df)} rows to raw_categories.csv")
saved_df.head(10)

## Reproducibility

Set `random_state` to control how data is shuffled between iterations. The same seed produces the same chunks each run (though LLM output still has some variance).

In [ ]:
raw_seeded = cat.explore(
    input_data=responses,
    description="Why did you move to your current residence?",
    api_key=api_key,
    user_model="gpt-4o-mini",
    iterations=3,
    divisions=4,
    random_state=42,
)

print(f"Seeded run: {len(raw_seeded)} raw, {len(set(raw_seeded))} unique")

## Rate Limiting with `chunk_delay`

Add a pause between processing each chunk. Useful for rate-limited providers (e.g., Google free tier at 5 RPM).

In [ ]:
raw_delayed = cat.explore(
    input_data=responses,
    description="Why did you move to your current residence?",
    api_key=api_key,
    user_model="gpt-4o-mini",
    iterations=2,
    divisions=4,
    chunk_delay=1.0,
)

print(f"With delay: {len(set(raw_delayed))} unique categories")

## Switching Providers

CatLLM auto-detects the provider from the model name. Just swap the model and API key.

In [ ]:
# Anthropic
raw_anthropic = cat.explore(
    input_data=responses,
    description="Why did you move to your current residence?",
    api_key="YOUR_ANTHROPIC_KEY",
    user_model="claude-sonnet-4-20250514",
    iterations=3,
    divisions=4,
)

print(f"Anthropic: {len(set(raw_anthropic))} unique categories")
for cat_name, freq in Counter(raw_anthropic).most_common(8):
    print(f"  {freq:2d}x  {cat_name}")

## Local Models with Ollama

Run exploration entirely locally using [Ollama](https://ollama.com/download). No API key needed.

Requires Ollama installed and running (`ollama serve`).

In [ ]:
raw_local = cat.explore(
    input_data=responses,
    description="Why did you move to your current residence?",
    api_key="not-needed",
    user_model="llama3.2",
    model_source="ollama",
    auto_download=True,
    iterations=3,
    divisions=4,
)

print(f"Ollama: {len(set(raw_local))} unique categories")
for cat_name, freq in Counter(raw_local).most_common(8):
    print(f"  {freq:2d}x  {cat_name}")

## Parameter Reference

| Parameter | Default | Effect |
|-----------|---------|--------|
| `description` | `""` | The survey question or description of the data. |
| `max_categories` | `12` | Maximum categories passed through to the extraction prompt. |
| `categories_per_chunk` | `10` | Max categories to extract from each chunk. |
| `divisions` | `12` | Chunks per iteration. More = smaller chunks = more focused extraction. |
| `iterations` | `8` | Number of passes over the data. More = more robust frequency counts. |
| `specificity` | `"broad"` | `"broad"` for high-level themes, `"specific"` for granular categories. |
| `focus` | `None` | Steer extraction toward a topic (e.g., "financial motivations"). |
| `research_question` | `None` | Academic context to guide extraction. |
| `creativity` | `None` | Temperature (0.0–1.0). Higher = more diverse categories. |
| `model_source` | `"auto"` | Provider — `"auto"`, `"openai"`, `"anthropic"`, `"ollama"`, etc. |
| `random_state` | `None` | Seed for reproducible chunking. |
| `chunk_delay` | `0.0` | Seconds between API calls (for rate limiting). |
| `filename` | `None` | Save raw categories to a CSV file. |
| `auto_download` | `False` | Auto-download missing Ollama models without prompting. |

**Tips:**
- For small datasets (<50 rows), reduce `divisions` to 3–5 so chunks aren't too small
- For saturation analysis, increase `iterations` to 15–20
- For cost-sensitive runs, use `iterations=2`, `divisions=3`

**Supported providers:** OpenAI, Anthropic, Google, Mistral, Perplexity, xAI, HuggingFace, Ollama (local).